Classic multilayer perceptron with two hidden layers to demonstrate the usage of the `Module` class.

In [1]:
import torch

class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(

            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

In [2]:
# Instantiate the NN class object
model = NeuralNetwork(50, 3) # 50 - number of inputs, 3 - number of outputs

In [3]:
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


Though not required, the `Sequential` class helps in keeping the sequence in the `__init__` class in order, and won't require specifying them again in the `forward` method. This way, the `self.layers` returns the the sequence of the layers.

In [5]:
# The number of trainable parameters in the model. These are the parameters with required_grad=True
# These trainable parameters are contained in the torch.nn.Linear layers which multiplies the inputs with the 
# weight matrix and add the bias vector to it. This is also refered to as fully-connected or feedforward

num_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
print("Total number of trainable model parameters:", num_params)

Total number of trainable model parameters: 2213


In [6]:
# From the print(model) output above, the first linear layer is at index position 0. 
# The weight parameter matrix can be accessed using

print(model.layers[0].weight)

Parameter containing:
tensor([[-0.1392, -0.0119,  0.0567,  ..., -0.0563,  0.1024,  0.1317],
        [ 0.0591, -0.0636, -0.0637,  ...,  0.0288, -0.0830,  0.0091],
        [-0.1162, -0.0392,  0.0440,  ..., -0.0641, -0.0926, -0.0020],
        ...,
        [-0.0340,  0.0232,  0.0575,  ..., -0.0888, -0.0838,  0.0309],
        [ 0.0147,  0.0665,  0.0281,  ...,  0.0761,  0.0895, -0.1207],
        [-0.1115,  0.0351,  0.1372,  ...,  0.0232, -0.0903, -0.1313]],
       requires_grad=True)


In [7]:
# To give us the weight matrix size
print(model.layers[0].weight.shape)

torch.Size([30, 50])


In [8]:
# The bias vector
print(model.layers[0].bias)

Parameter containing:
tensor([ 0.0987, -0.0891,  0.0114,  0.0129,  0.0126, -0.0775, -0.0172,  0.0035,
         0.1271, -0.1124,  0.0073,  0.0094, -0.0064, -0.1160,  0.1344, -0.0645,
        -0.1388, -0.0691,  0.0580,  0.1355,  0.0121, -0.0519, -0.0205, -0.0031,
         0.0538, -0.0183, -0.1292, -0.0257,  0.1170, -0.1228],
       requires_grad=True)


In [9]:
# The bias vector shape
print(model.layers[0].bias.shape)

torch.Size([30])


Manual seed to allow reproducibility in the initialization of the model weights. 
Model weights are initialized with small random numbers and would be different at every time of the instantiation. This is to avoid computing the same operations, and allow the network to learn complex mapping from inputs to outputs

In [10]:
torch.manual_seed(123)

model = NeuralNetwork(50, 3)
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


In [11]:
torch.manual_seed(123)  # to keep the weight initialization constant

X = torch.rand((1, 50)) 
# toy input, the 50 is to conform with the 50-dimensional feature vector expected by the model
out = model(X) # Input fed into the model, and the model executes the forward pass model.
print(out)

tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


The three output numbers are assigned to the three output nodes. The `grad_fn` parameter represents the last used function to compute the computational graph variable. From the `<AddmmBackward0>` value, it means the tensor being inspected was created using matrix multiplication and addition operation. This info is used when computing the gradients during backpropagation.


For instances when the constructing the computational graph is considered wasteful (when the model is being used for prediction and not training), the `torch.no_grad()` context manager is used to tell PyTorch it does not need to keep track of the gradients

In [12]:
with torch.no_grad():
    out = model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]])


PyTorch commonly returns the output of the last layer without passing them to a nonlinear activation function. This is because the commonly used loss functions combine softmax (or sigmoid for binary classification) operation with the negative log-likelihood loss in a single class. This gives numerical efficiency and stability. The softmax function can be called explicitly to compute the class-membership probabilities for our prediction.

In [14]:
# The values are apprx summed to be 1.
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)

tensor([[0.3113, 0.3934, 0.2952]])
